# The time value of money, in code

**pyportfolios.com** · [/research/time-value-of-money](https://pyportfolios.com/research/time-value-of-money) · US Treasury yields, Dec 2021 – Jan 2023 · NumPy · Pandas · yfinance

Every valuation is the same gesture: take cash arriving later, ask what it is worth
now. In this notebook we do it with *real* rates — the US Treasury curve straight
off Yahoo Finance, through the most violent repricing of money in four decades
(calendar 2022). We

1. download the 13-week, 5y, 10y and 30y Treasury yields,
2. watch the price of time explode through 2022,
3. build discount curves DF(t) from the observed zeros,
4. compare compounding conventions at the real 10y rate, and
5. price a 5% coupon bond off both curves — and see rate risk fall out.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams["figure.figsize"] = (10, 5)

## 1 · Real rates, from the tape

Yahoo's Treasury yield indices are quoted in percent (^TNX = 3.88 means 3.88%).
Four points on the curve: 13 weeks, 5, 10 and 30 years.

In [ ]:
TICKERS = ["^IRX", "^FVX", "^TNX", "^TYX"]   # 13w, 5y, 10y, 30y
MATS    = [0.25, 5, 10, 30]

ylds = yf.download(TICKERS, start="2021-12-01", end="2023-01-10",
                   auto_adjust=True, progress=False)["Close"][TICKERS].dropna()

ylds.plot(title="US Treasury yields through 2022 (percent)")
plt.ylabel("yield (%)");

jan = ylds.loc["2022-01-03"]
dec = ylds.loc["2022-12-30"]
print(pd.DataFrame({"3 Jan 2022": jan.values, "30 Dec 2022": dec.values},
                   index=["3m", "5y", "10y", "30y"]).round(2))

The 13-week bill went from essentially free money to over 4% inside a year. Money
re-acquired its time stamp in 2022 — which makes it the perfect sample for this topic.

## 2 · From yields to a discount curve

A zero rate y(t) and continuous compounding give the discount factor
$DF(t) = e^{-y(t)\,t}$ — the price, today, of one dollar delivered at t. We
interpolate the four observed zeros linearly in yield and draw DF(t) for both
ends of 2022.

In [ ]:
def zero_curve(yields_pct):
    y_dec = np.asarray(yields_pct, dtype=float) / 100
    return lambda t: np.interp(t, MATS, y_dec)

y_jan, y_dec_ = zero_curve(jan.values), zero_curve(dec.values)

t = np.linspace(0, 30, 121)
plt.plot(t, np.exp(-y_jan(np.maximum(t, 0.25)) * t), lw=2, label="curve of 3 Jan 2022")
plt.plot(t, np.exp(-y_dec_(np.maximum(t, 0.25)) * t), lw=2, label="curve of 30 Dec 2022")
plt.xlabel("maturity t (years)"); plt.ylabel("DF(t)")
plt.title("Discount factor by maturity — one dollar, delivered later"); plt.legend();

print(f"DF(30y), Jan 2022: {np.exp(-y_jan(30) * 30):.4f}")
print(f"DF(30y), Dec 2022: {np.exp(-y_dec_(30) * 30):.4f}")

## 3 · Present value, concretely

$100 arriving in 10 years, discounted at the observed 10y zero:

In [ ]:
for name, fn in [("Jan 2022", y_jan), ("Dec 2022", y_dec_)]:
    y10 = fn(10)
    print(f"{name}: 10y zero = {y10:.2%}  ->  PV of $100 in 10y = ${100 * np.exp(-y10 * 10):.2f}")

## 4 · Compounding conventions

The same quoted rate means different things depending on how often it compounds.
At the real end-2022 10y rate, here is DF(10) under each convention — same
economics, different units:

In [ ]:
r10 = float(y_dec_(10))
print(f"10y zero rate: {r10:.4%}\n")
for name, df in [("annual",     (1 + r10) ** -10),
                 ("semi-annual",(1 + r10 / 2) ** -20),
                 ("monthly",    (1 + r10 / 12) ** -120),
                 ("continuous", np.exp(-r10 * 10))]:
    print(f"{name:>12}: DF(10) = {df:.4f}")

## 5 · Valuing a cashflow stream

Price a 5% annual-coupon bond (par 100, 5 years) off the real end-2022 curve.
The valuation is one line — flows dotted against discount factors.

In [ ]:
cf = pd.DataFrame({"t": [1, 2, 3, 4, 5], "flow": [5, 5, 5, 5, 105]})

cf["rate"] = y_dec_(cf["t"].to_numpy(dtype=float))
cf["df"]   = np.exp(-cf["rate"] * cf["t"])
cf["pv"]   = cf["flow"] * cf["df"]
print(cf.round(4))
print(f"\nBond price off the Dec 2022 curve: {cf['pv'].sum():,.2f}")

# same bond, one year earlier
cf["pv_jan"] = cf["flow"] * np.exp(-y_jan(cf["t"].to_numpy(dtype=float)) * cf["t"])
print(f"Bond price off the Jan 2022 curve: {cf['pv_jan'].sum():,.2f}")

## Takeaways

- A discount factor is a *price* — of one dollar delivered later. Everything else
  (PV, bond prices, swap legs) is a dot product against those prices.
- Conventions change the arithmetic, not the economics: know which one a quote uses.
- The 2022 curve shift repriced the same five cashflows by several points of par.
  That sensitivity is duration — the next tutorial in the chain.

*© pyportfolios.com — runnable companion to the article. Data: Yahoo Finance via yfinance.*